# Bài 18/25 · LeetCode #104 — Maximum Depth of Binary Tree

**Mức độ:** Easy · **Chủ đề:** Tree DFS · Recursion · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/maximum-depth-of-binary-tree/](https://leetcode.com/problems/maximum-depth-of-binary-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Tính số node trên đường đi **dài nhất** từ gốc đến lá của cây nhị phân. Cây rỗng có độ sâu 0.

### Ví dụ quan trọng

`[3,9,20,None,None,15,7] → 3`; `[] → 0`; `[1,2] → 2`.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [1]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. `TreeNode` thật sự là gì? Học từ một nút đơn

**Binary Tree (cây nhị phân)** gồm các nút (*node*). Mỗi nút lưu đúng ba thông tin:
`val` (giá trị), `left` (tham chiếu đến con trái), `right` (tham chiếu đến con phải).

Quan trọng: `left`/`right` **không phải** là vị trí trong mảng; đó là liên kết đến một **đối tượng** `TreeNode` khác.
`None` nghĩa là *không có nút*. Một cây rỗng được biểu diễn bởi `root = None`.

Ví dụ cây `1` có con trái `2` và con phải `3`:

```text
      1
     / \
    2   3
```

Nhìn vào code dựng cây dưới đây, hãy dự đoán từng phép in trước khi chạy.

In [2]:
root_demo = TreeNode(1)
root_demo.left = TreeNode(2)
root_demo.right = TreeNode(3)
print("Gốc:", root_demo.val)
print("Con trái:", root_demo.left.val)
print("Con phải:", root_demo.right.val)
print("Con trái của nút 2:", root_demo.left.left)  # None

Gốc: 1
Con trái: 2
Con phải: 3
Con trái của nút 2: None


### 2.2. Tham chiếu (*reference*) và giá trị khác nhau thế nào?

`a = root_demo.left` **không sao chép** một nút mới: `a` và `root_demo.left` là hai tên cùng chỉ đến một đối tượng.
Vì vậy `a.val = 99` sẽ thay đổi giá trị trong cây gốc.

- `a is b`: hỏi **có cùng một đối tượng** hay không.
- `a.val == b.val`: hỏi **hai giá trị bằng nhau** hay không.
- Hai nút khác nhau có thể cùng giá trị; với so sánh cây, ta cần xét cả cấu trúc, không chỉ xét một con số.

Phần này đặc biệt quan trọng khi gắn nút con trong giải pháp chính.

In [3]:
x_demo = TreeNode(7)
y_demo = TreeNode(7)
z_demo = x_demo
print("Giá trị bằng nhau:", x_demo.val == y_demo.val)  # True
print("Là một đối tượng:", x_demo is y_demo)        # False
print("Chỉ đến cùng nút:", x_demo is z_demo)       # True
z_demo.val = 8
print("Giá trị của x sau khi đổi qua z:", x_demo.val)

Giá trị bằng nhau: True
Là một đối tượng: False
Chỉ đến cùng nút: True
Giá trị của x sau khi đổi qua z: 8


### 2.3. `None`, `is None` và thứ tự kiểm tra an toàn

Với `node = None`, viết `node.val` sẽ gây `AttributeError`: không thể truy cập `val` của vật không tồn tại.
Cách an toàn: kiểm tra `node is None` **trước**, rồi mới dùng `node.val`.

Trong Python, biểu thức `A and B` có **short-circuit** (đoản mạch): nếu `A` là `False` thì Python **không đánh giá** `B` nữa.
Tương tự, với `A or B`, nếu `A` là `True` thì không cần tính `B`.
Đây là lý do ta có thể xử lý trường hợp rỗng trước khi truy cập thuộc tính và tránh nhánh đệ quy vô ích.

In [4]:
nothing = None
print("Là nút rỗng?", nothing is None)
print("Không phải nút rỗng?", nothing is not None)
print("Đọc an toàn:", nothing.val if nothing is not None else "không có nút")
leaf_demo = TreeNode(42)
print("Lá có hai con rỗng:", leaf_demo.left is None and leaf_demo.right is None)

Là nút rỗng? True
Không phải nút rỗng? False
Đọc an toàn: không có nút
Lá có hai con rỗng: True


### 2.4. Hiểu danh sách level-order và vì sao phải giữ `None`

LeetCode thường hiển thị cây theo thứ tự từng tầng, trái trước phải sau (*level-order*).
Trong notebook, hàm `tree(values)` dựng cây từ dạng ấy để dễ thử, còn `show_tree(root)` đưa cây về danh sách để đọc.
**Đừng coi list đó là định nghĩa của cây**: thuật toán nộp LeetCode nhận `TreeNode`, không nhận danh sách.

Ví dụ `[1,2,None,3]` là: gốc 1 có con trái 2, nút 2 có con trái 3.
Nó khác `[1,None,2,3]`: gốc 1 có con phải 2, nút 2 có con trái 3.
Dù phần tử `3` xuất hiện cuối cả hai danh sách, vị trí con trỏ khác nhau hoàn toàn.

In [5]:
shape_a = tree([1, 2, None, 3])
shape_b = tree([1, None, 2, 3])
print("A:", show_tree(shape_a), "gốc trái:", shape_a.left)
print("B:", show_tree(shape_b), "gốc trái:", shape_b.left)
print("A.gốc.phải:", shape_a.right, "B.gốc.phải:", shape_b.right)
assert show_tree(shape_a) != show_tree(shape_b)

A: [1, 2, None, 3] gốc trái: TreeNode(2)
B: [1, None, 2, 3] gốc trái: None
A.gốc.phải: None B.gốc.phải: TreeNode(2)


### 2.5. Đệ quy (*recursion*) từ con số 0: hàm gọi lại chính nó

Đệ quy có hai mảnh **bắt buộc**:

1. **Base case** (trường hợp dừng): đơn giản đến mức trả lời ngay, không gọi tiếp.
2. **Recursive step** (bước rút gọn): gọi lại hàm trên bài toán nhỏ hơn, rồi dùng kết quả ấy.

**Call stack (ngăn xếp lời gọi)** giữ lại vị trí mà mỗi lần gọi đang chờ `return`.
Ví dụ `countdown(3)` cần đi `3 → 2 → 1 → 0` rồi quay trở ra.
Trên cây, bước nhỏ hơn tự nhiên là cây con `node.left` hoặc `node.right`.

Kí hiệu chiều cao `h`: một đường từ gốc xuống lá dài tối đa `h` nút, nên tầng gọi lồng nhau có thể đến `h`.
Đệ quy không tự biến mất bộ nhớ: mỗi lần gọi đang chờ chiếm một phần call stack.

In [6]:
def countdown_demo(n, level=0):
    print("  " * level + f"vào n={n}")
    if n == 0:
        print("  " * level + "đã đến base case")
        return 0
    result = 1 + countdown_demo(n - 1, level + 1)
    print("  " * level + f"trở lại n={n}, trả {result}")
    return result

assert countdown_demo(3) == 3

vào n=3
  vào n=2
    vào n=1
      vào n=0
      đã đến base case
    trở lại n=1, trả 1
  trở lại n=2, trả 2
trở lại n=3, trả 3


### 2.6. `Optional[TreeNode]` và `return` là gì?

`Optional[TreeNode]` (trong `typing`) mô tả kiểu dữ liệu có thể là một `TreeNode` **hoặc** `None`.
Đây là **type hint** (gợi ý kiểu), không làm Python tự kiểm tra cây hợp lệ trong lúc chạy.

`return` *trả kết quả về nơi gọi và kết thúc lượt chạy hiện tại*. `print` chỉ đưa thông tin lên màn hình;
một hàm chỉ `print(True)` mà không `return True` sẽ trả về `None` (không đúng kiểu `bool` mà nhiều bài đòi).

Vòng lặp `for`, phép `max`, phép `abs`, danh sách `[]`, biểu thức `x is None`, phép truy cập `.left` / `.right`
đều sẽ được giải thích lại ngay tại chỗ cần dùng ở phần sau.

In [7]:
def check_node_demo(node: Optional[TreeNode]) -> bool:
    if node is None:
        return False
    return node.val > 0

print("Kiểm tra None:", check_node_demo(None))
print("Kiểm tra nút 5:", check_node_demo(TreeNode(5)))
print("Kiểm tra nút -2:", check_node_demo(TreeNode(-2)))

Kiểm tra None: False
Kiểm tra nút 5: True
Kiểm tra nút -2: False


## 3. Python nền tảng — công cụ cần hiểu trước khi giải

Recursion có **base case**: với `None`, độ sâu bằng 0; với node còn lại, lấy `max` của độ sâu hai nhánh.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [8]:
print('max(2,3) =',max(2,3))
root=tree([3,9,20,None,None,15,7])
print('Gốc:',root.val,'trái:',root.left.val,'phải:',root.right.val)
print('Con trái là lá?',root.left.left is None and root.left.right is None)

max(2,3) = 3
Gốc: 3 trái: 9 phải: 20
Con trái là lá? True


### 3.1. Độ sâu tính theo **số nút**, không theo số cạnh

Với #104, **maximum depth** là số **nút** trên đường đi dài nhất từ gốc tới một **lá**.
Lá (*leaf*) là nút không có con trái lẫn con phải.

| Hình dạng | Depth |
|---|---:|
| Cây rỗng `None` | 0 |
| Chỉ có nút `7` | 1 |
| `7 → con trái 3` | 2 |
| `7 → trái 3 → phải 2` | 3 |

Trong một cây 3 nút nối thành dây, có **2 cạnh** nhưng **3 nút**.
Đây là lý do công thức `1 + max(...)`: số `1` chính là nút hiện tại.
Không được cộng `1` cho `None`, vì chỗ trống không phải một nút.

In [9]:
depth_examples = [
    ([], 0),
    ([7], 1),
    ([7, 3], 2),
    ([7, 3, None, None, 2], 3),
]
for values, expected_depth in depth_examples:
    print(f"{values} -> depth dự kiến {expected_depth}")
    print("  Cây sau dựng:", show_tree(tree(values)))

[] -> depth dự kiến 0
  Cây sau dựng: []
[7] -> depth dự kiến 1
  Cây sau dựng: [7]
[7, 3] -> depth dự kiến 2
  Cây sau dựng: [7, 3]
[7, 3, None, None, 2] -> depth dự kiến 3
  Cây sau dựng: [7, 3, None, None, 2]


### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Từ gốc, duyệt theo từng tầng BFS, **đếm số tầng** đến khi hàng đợi rỗng. Cách này trực giác như đi qua từng hàng của cây.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [10]:
def baseline(root):
    if root is None:return 0
    q=deque([root]);level=0
    while q:
        level+=1
        for _ in range(len(q)):
            node=q.popleft()
            if node.left:q.append(node.left)
            if node.right:q.append(node.right)
    return level

In [11]:
for a in [[],[1],[3,9,20,None,None,15,7]]:
    print(a,'->',baseline(tree(a)))

[] -> 0
[1] -> 1
[3, 9, 20, None, None, 15, 7] -> 3


### 4.1. Giải thích baseline: đi theo từng đường xuống lá

Người mới thường nghĩ: liệt kê mọi đường từ gốc tới lá, đếm số nút mỗi đường, rồi lấy số lớn nhất.
Cách đó đúng, dễ hiểu, nhưng phải **giữ danh sách đường đi** nếu ta muốn lưu tất cả các đường.
Với bài chỉ hỏi **một con số**, việc lưu mọi đường đi là thừa.

Ta có thể viết code baseline trả về danh sách độ dài đường đi đến các lá. Sau đó `max(...)` lấy số lớn nhất.
Đây là phiên bản phục vụ **giải thích**, không phải code nên nộp LeetCode khi có công thức đơn giản hơn.

In [12]:
def depths_of_root_to_leaf_paths(root):
    if root is None:
        return []
    if root.left is None and root.right is None:
        return [1]
    subpath_lengths = (depths_of_root_to_leaf_paths(root.left)
                       + depths_of_root_to_leaf_paths(root.right))
    return [length + 1 for length in subpath_lengths]

sample_depth_tree = tree([3, 9, 20, None, None, 15, 7])
print("Các độ dài đường đi:", depths_of_root_to_leaf_paths(sample_depth_tree))
assert max(depths_of_root_to_leaf_paths(sample_depth_tree)) == 3

Các độ dài đường đi: [2, 3, 3]


### 4.2. Rút ra “công thức quy hoạch trên cây” từ hai bài nhỏ

Giả sử ta đã biết `depth(left)` và `depth(right)`.

Nếu đứng ở nút hiện tại, muốn đi xa nhất xuống lá, ta chọn **nhánh dài hơn**,
rồi cộng thêm **1 nút hiện tại**:

`depth(node) = 1 + max(depth(node.left), depth(node.right))`

**Trường hợp dừng:** `depth(None)=0`.

Ví dụ nút gốc có con trái sâu 1, con phải sâu 2 ⇒ `depth(gốc)=1+max(1,2)=3`.
Công thức không cần biết tên nút hoặc giá trị `val`; chỉ cần cấu trúc cây.
Đó là **bottom-up**: tính thông tin từ cây con rồi trả lên cha.

In [13]:
left_depth = 1
right_depth = 2
my_depth = 1 + max(left_depth, right_depth)
print("Chiều sâu trái:", left_depth)
print("Chiều sâu phải:", right_depth)
print("Chiều sâu của cha:", my_depth)
assert my_depth == 3

Chiều sâu trái: 1
Chiều sâu phải: 2
Chiều sâu của cha: 3


## 5. Vì sao cần chuyển sang kỹ thuật chính?

Có thể tính bằng công thức trên cây con: `depth(node) = 1 + max(depth(left),depth(right))` và `depth(None)=0`. Công thức tự mô tả một bài toán nhỏ hơn.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Hàm đệ quy trả **độ sâu**. Node lá trả 1 vì hai nhánh `None` đều có độ sâu 0. `max()` chọn đường dài hơn, không phải cộng độ sâu hai nhánh.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

In [14]:
def solve(root: Optional[TreeNode]) -> int:
    def depth(node):
        if node is None:return 0
        return 1+max(depth(node.left),depth(node.right))
    return depth(root)

### 6.1. Đọc code đệ quy **từng dòng**, không bỏ qua nút rỗng

`def height(node)` là hàm trả về maximum depth **của cây con hiện tại**.

1. Nếu `node is None`, trả `0`: không có nút nào.
2. Gọi đệ quy lấy chiều sâu cây con trái `L`.
3. Gọi đệ quy lấy chiều sâu cây con phải `R`.
4. Kết luận: `1 + max(L,R)`.

Không cần `if leaf`: một lá có hai con `None`, tự động trả `1+max(0,0)=1`.
Đây là ưu điểm của việc chọn đúng base case.

Đừng nhầm với **minimum depth** (#111): bài minimum có nút chỉ một con thì không được chọn nhánh trống
có độ dài `0`; còn maximum depth dùng `max` nên nhánh trống không gây vấn đề.

In [15]:
print("Lá:", 1 + max(0, 0))
print("Cha có trái sâu 2 và phải rỗng:", 1 + max(2, 0))
print("Cha có trái sâu 1, phải sâu 3:", 1 + max(1, 3))
assert solve(tree([7])) == 1
assert solve(tree([7, 3, None, 2])) == 3

Lá: 1
Cha có trái sâu 2 và phải rỗng: 3
Cha có trái sâu 1, phải sâu 3: 4


## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Node | sâu trái | sâu phải | Trả về |
|---|---:|---:|---:|
| `9` (lá) | 0 | 0 | 1 |
| `15` (lá) | 0 | 0 | 1 |
| `20` | 1 | 1 | 2 |
| `3` (gốc) | 1 | 2 | **3** |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [16]:
def trace_depth(node,level=0):
    if node is None:
        print('  '*level,'None -> 0'); return 0
    print('  '*level,'vào node',node.val)
    l=trace_depth(node.left,level+1)
    r=trace_depth(node.right,level+1)
    ans=1+max(l,r)
    print('  '*level,f'node={node.val}; trái={l}; phải={r} -> {ans}')
    return ans
trace_depth(tree([3,9,20,None,None,15,7]))

 vào node 3
   vào node 9
     None -> 0
     None -> 0
   node=9; trái=0; phải=0 -> 1
   vào node 20
     vào node 15
       None -> 0
       None -> 0
     node=15; trái=0; phải=0 -> 1
     vào node 7
       None -> 0
       None -> 0
     node=7; trái=0; phải=0 -> 1
   node=20; trái=1; phải=1 -> 2
 node=3; trái=1; phải=2 -> 3


3

### 7.1. Trace bằng tay: tính *sau khi* cây con trả kết quả

Cây `[3,9,20,None,None,15,7]` có:

```text
        3
       / \
      9  20
        /  \
       15   7
```

Thứ tự *return* theo đệ quy trái-trước-phải, với các nhánh rỗng được xem là 0:

| Nút | Depth trái | Depth phải | Depth nút này |
|---:|---:|---:|---:|
| 9 | 0 | 0 | 1 |
| 15 | 0 | 0 | 1 |
| 7 | 0 | 0 | 1 |
| 20 | 1 | 1 | 2 |
| 3 | 1 | 2 | **3** |

Trace không phải một phép cộng đều đặn theo giá trị `val`: nút `20` không làm kết quả thành 20.
Chính **chiều cao của hai cây con** mới quyết định kết quả.

### 7.2. Cách 2 — BFS: đi theo từng *tầng*, không đi sâu trước

*Breadth-First Search (BFS)* thăm cây theo từng tầng. Dùng queue (`deque`) để lấy nút ở đầu hàng,
thêm các con ở cuối. Kỹ thuật `for _ in range(len(queue))` xử lý **đúng số nút hiện có ở đầu tầng**;
con được thêm ở vòng lặp sẽ dành cho tầng tiếp theo.

Mỗi lần xử lý xong **một tầng**, tăng `depth += 1`. Khi queue rỗng, `depth` chính là số tầng của cây.
Cách này cũng `O(n)` thời gian nhưng dùng queue rộng `O(w)` với `w` là số nút tối đa ở một tầng.

In [17]:
def max_depth_bfs(root):
    if root is None:
        return 0
    q = deque([root])
    depth = 0
    while q:
        width = len(q)   # chỉ số nút của tầng hiện tại
        for _ in range(width):
            node = q.popleft()
            if node.left is not None:
                q.append(node.left)
            if node.right is not None:
                q.append(node.right)
        depth += 1
    return depth

for data in [[],[1],[1,2,3],[3,9,20,None,None,15,7]]:
    print(data, '->', max_depth_bfs(tree(data)))

[] -> 0
[1] -> 1
[1, 2, 3] -> 2
[3, 9, 20, None, None, 15, 7] -> 3


### 7.3. Trace BFS: mỗi dòng là một tầng đã xử lý

Ghi rõ **queue trước** và **queue sau** khi xử lý tầng.
Nếu bạn vô tình dùng `while q` bên trong thay vì `for range(width)`, các nút mới thêm sẽ bị xử lý ngay,
khiến toàn bộ cây có thể được đếm thành đúng một tầng.
Vì vậy biến `width` lưu số nút đầu vòng lặp là cực kỳ quan trọng.

In [18]:
def trace_depth_bfs(root):
    if root is None:
        print("Cây rỗng: 0 tầng")
        return 0
    q = deque([root]); depth = 0
    while q:
        print("Đầu tầng", depth+1, ":", [n.val for n in q])
        width = len(q)
        for _ in range(width):
            node = q.popleft()
            if node.left is not None: q.append(node.left)
            if node.right is not None: q.append(node.right)
        depth += 1
        print("  Cuối tầng: queue tiếp theo =", [n.val for n in q])
    return depth

assert trace_depth_bfs(tree([3,9,20,None,None,15,7])) == 3

Đầu tầng 1 : [3]
  Cuối tầng: queue tiếp theo = [9, 20]
Đầu tầng 2 : [9, 20]
  Cuối tầng: queue tiếp theo = [15, 7]
Đầu tầng 3 : [15, 7]
  Cuối tầng: queue tiếp theo = []


## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Đường đi sâu nhất từ `node` bắt đầu bằng chính `node`, rồi đi vào đúng **một** trong hai nhánh. Nếu chiều sâu hai cây con đúng theo giả thiết quy nạp, cộng 1 với độ sâu lớn hơn cho đúng độ sâu của cây hiện tại.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [19]:
from typing import Optional
class Solution:
    def maxDepth(self, root: Optional[TreeNode]):
        def depth(node):
            if node is None:return 0
            return 1+max(depth(node.left),depth(node.right))
        return depth(root)

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [20]:
for a,expected in [([],0),([1],1),([1,2],2),([3,9,20,None,None,15,7],3),([1,None,2,None,3],3)]:
    assert solve(tree(a))==expected
    assert Solution().maxDepth(tree(a))==expected
print('PASS 5 ca độ sâu')

PASS 5 ca độ sâu


### 10.1. So sánh đệ quy, BFS và baseline trên nhiều hình dạng cây

Test gốc rỗng, chỉ một lá, cây lệch trái, cây lệch phải, cây đầy đủ, cây không đầy đủ,
cây có nhiều giá trị bằng nhau và giá trị âm. **Giá trị không được ảnh hưởng đến chiều sâu**.
Có thể tự thử cây lệch 100 nút, nhưng đừng vượt quá giới hạn đệ quy Python trong demo.

In [21]:
max_depth_cases = [
    ([], 0), ([1], 1), ([3,9,20,None,None,15,7], 3),
    ([1,2,None,3,None,4], 4), ([1,None,2,None,3], 3),
    ([1,2,3,4,5,6,7], 3), ([1,2,3,None,4,None,5], 3),
    ([0,0,0,0,0,0,0], 3), ([-100,-1,None,-2], 3),
]
for arr, expected in max_depth_cases:
    t = tree(arr)
    assert solve(t) == expected, arr
    assert max_depth_bfs(t) == expected, arr
    assert Solution().maxDepth(t) == expected, arr
    assert (max(depths_of_root_to_leaf_paths(t)) if t else 0) == expected
print(f"PASS {len(max_depth_cases)} ca × 4 cách")

PASS 9 ca × 4 cách


### 10.2. Thuộc tính: đổi mọi `val` không làm depth đổi

Ta có thể dựng hai cây **cùng hình dáng** nhưng giá trị rất khác, kể cả giá trị âm và trùng lặp.
Vì công thức `1+max(L,R)` không đọc `node.val`, kết quả depth phải bằng nhau.
Đây là cách test giúp bắt lỗi giải nhầm “depth” thành phép toán trên giá trị.

In [22]:
a_shape = tree([4,8,12,16,None,None,20])
b_shape = tree([-4,-8,-12,-16,None,None,-20])
print("Cây A:", show_tree(a_shape))
print("Cây B:", show_tree(b_shape))
assert solve(a_shape) == solve(b_shape) == 3
print("PASS property: cùng cấu trúc thì depth như nhau")

Cây A: [4, 8, 12, 16, None, None, 20]
Cây B: [-4, -8, -12, -16, None, None, -20]
PASS property: cùng cấu trúc thì depth như nhau


## 11. Phân tích Time/Space Complexity

`n` node; DFS thăm mỗi node một lần: **Time O(n)**; **Space O(h)** cho recursion. BFS baseline **O(n)** time, tối đa O(w) bộ nhớ (w = số node lớn nhất ở một tầng).

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ `depth(left)+depth(right)`: đó là cộng node của hai nhánh, không phải một đường đi.
- ❌ Trả 1 khi `root is None`: sai với cây rỗng.
- ❌ Bỏ qua chiều sâu của nhánh còn lại.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Code sai minh họa: không tính nút hiện tại

Sai lầm phổ biến là viết `return max(height(left), height(right))` mà quên `+1`.
Với lá, hai con rỗng đều trả `0`, nên lá cũng bị coi là có depth `0`, và cả cây trả `0`.

Một lỗi khác: đặt `height(None)=1`. Với lá thì depth thành `2`, tức là cộng sai cho chỗ trống.
**Quy ước nhất quán:** `None → 0`, `node → 1 + max(các con)`.
Code minh họa dưới chỉ cho thấy đầu ra sai, không gây exception.

In [23]:
def wrong_depth_without_self(node):
    if node is None:
        return 0
    return max(wrong_depth_without_self(node.left),
               wrong_depth_without_self(node.right))

example = tree([3,9,20,None,None,15,7])
print("Sai: quên +1:", wrong_depth_without_self(example))
print("Đúng:", solve(example))
assert wrong_depth_without_self(example) != solve(example)

Sai: quên +1: 0
Đúng: 3


## 13. Nhận diện dạng bài và vận dụng

Đề hỏi chiều sâu/chiều cao của cây: tìm công thức `1 + max(...)` sau khi xác định đúng ý nghĩa giá trị trả về của mỗi lời gọi.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Thử cây lệch toàn bộ về bên phải để thấy kết quả vẫn là **số node trên đường đi**, không phải độ sâu tính theo cạnh.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [24]:
root=tree([1,None,2,None,3,None,4]);print('maxDepth=',solve(root))

maxDepth= 4


### 14.1. Khi nào nên dùng DFS, khi nào nên dùng BFS?

| Đặc điểm | Đệ quy DFS | BFS theo tầng |
|---|---|---|
| Trực giác | Chiều sâu cây con → chiều sâu cha | Mỗi vòng while → thêm 1 tầng |
| Thời gian | `O(n)` | `O(n)` |
| Bộ nhớ phụ | `O(h)` call stack | `O(w)` queue (chiều rộng lớn nhất) |
| Cây rất lệch | Call stack sâu; Python có giới hạn đệ quy | Queue nhỏ, không cần recursion |
| Cây rất rộng | Call stack tương đối nông | Queue có thể nhiều nút |

**Pattern:** khi đề hỏi một giá trị tổng hợp từ hai cây con, đệ quy trả thông tin lên cha rất tự nhiên.
Khi đề hỏi *số tầng* hoặc xử lý từng tầng, BFS trực quan hơn.
Trong tình huống duyệt cây thực tế phải cân nhắc chiều cao và chiều rộng, không khẳng định “DFS luôn tốt hơn”.

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Cho `r=tree([3,9,20,None,None,15,7])`. Dự đoán `print(r.val, r.left.val, r.right.val)` và `print(max(1,2))`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [25]:
r=tree([3,9,20,None,None,15,7])
print(r.val,r.left.val,r.right.val)
print(max(1,2))

3 9 20
2


### Bài 2 — Trace bằng tay (hiểu bản chất)

Tính tay độ sâu lớn nhất của `[3,9,20,None,None,15,7]`: độ sâu cây con gốc `9` là bao nhiêu, của gốc `20` là bao nhiêu? Từ đó suy ra đáp án.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_max_depth(root)` đệ quy với quy tắc `1 + max(độ sâu trái, độ sâu phải)`; cây rỗng có độ sâu 0.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [26]:
def exercise_max_depth(root):
    # TODO: base case cây rỗng.
    # TODO: lấy max 2 lời gọi trên 2 cây con.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Viết `exercise_max_depth_stack(root)` duyệt DFS không đệ quy: mỗi phần tử stack là `(node, depth)`.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [27]:
def exercise_max_depth_stack(root):
    # TODO: xử lý cây rỗng, cập nhật độ sâu tốt nhất.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: giải bài bằng **BFS** (`deque`): mỗi vòng ngoài xử lý đúng một tầng; đếm số tầng đã đi qua.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [28]:
def exercise_max_depth_bfs(root):
    # TODO: deque ban đầu chứa gốc; theo dõi độ dài của mỗi tầng.
    raise NotImplementedError

### Bài 6 — Đếm số lá của một cây bất kỳ

Tự viết `exercise_count_leaves(root)` trả **số lá** (nút có cả hai con `None`).
Ví dụ cây `[3,9,20,None,None,15,7]` có 3 lá: `9`, `15`, `7`.

**Gợi ý cấp 1:** cây rỗng không có lá; một nút không có con chính là một lá.

<details><summary>Gợi ý cấp 2 — điều kiện</summary><code>node.left is None and node.right is None</code>.</details>

<details><summary>Gợi ý cấp 3 — tổng</summary>Nếu không phải lá, số lá của node = số lá cây trái + số lá cây phải.</details>

In [29]:
def exercise_count_leaves(root):
    # TODO: None => 0; node không có con => 1;
    # node còn con => cộng số lá của hai cây con.
    raise NotImplementedError

# Sau khi làm: assert exercise_count_leaves(tree([3,9,20,None,None,15,7])) == 3

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Độ sâu tính theo **số node** từ gốc tới lá, không phải số cạnh.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

DFS: mang theo `depth`; BFS: mỗi lượt quét hết `len(q)` node của tầng hiện tại.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Base case `root is None -> 0`; đệ quy thêm `1`; stack khởi đầu `(root,1)`; BFS tăng `depth` sau khi hết một tầng.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Hai dòng: `3 9 20` và `2`. `max` chọn **nhánh dài hơn**.

### Bài 2 — Trace tay

Gốc `9` có độ sâu `1`; gốc `20` có độ sâu `2`; gốc chung thêm 1 nên `1 + max(1,2) = 3`.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [30]:
def answer_max_depth(root):
    if root is None: return 0
    return 1+max(answer_max_depth(root.left),answer_max_depth(root.right))

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [31]:
def answer_max_depth_stack(root):
    if root is None: return 0
    stack=[(root,1)]
    best=0
    while stack:
        node,depth=stack.pop()
        best=max(best,depth)
        if node.left: stack.append((node.left,depth+1))
        if node.right: stack.append((node.right,depth+1))
    return best

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [32]:
def answer_max_depth_bfs(root):
    if root is None: return 0
    q=deque([root]);depth=0
    while q:
        for _ in range(len(q)):
            node=q.popleft()
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        depth+=1
    return depth

### Đáp án bài 6 — số lá tính bằng đệ quy

Đây cũng là mẫu **bottom-up**: đếm kết quả của hai cây con rồi cộng lại.
Khác với depth, ta cộng số lá; không lấy `max`.
Hãy so sánh hai công thức: `leaves = left + right`, `depth = 1 + max(left, right)`.

In [33]:
def answer_count_leaves(root):
    if root is None:
        return 0
    if root.left is None and root.right is None:
        return 1
    return answer_count_leaves(root.left) + answer_count_leaves(root.right)

assert answer_count_leaves(None) == 0
assert answer_count_leaves(TreeNode(7)) == 1
assert answer_count_leaves(tree([3,9,20,None,None,15,7])) == 3
print("PASS bài 6")

PASS bài 6


### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [34]:
for arr,expected in [([],0),([1],1),([3,9,20,None,None,15,7],3),([1,2,None,3,None,4],4)]:
    for f in (answer_max_depth,answer_max_depth_stack,answer_max_depth_bfs):
        assert f(tree(arr))==expected
print("PASS: đáp án 3 bài coding của phần luyện tập")

PASS: đáp án 3 bài coding của phần luyện tập


### Tự đánh giá #104 — giải lại mà không nhìn code

1. Vì sao cây một nút có depth 1, không phải 0?
2. Base case cho `None` trả gì? Vì sao?
3. Trace tại gốc với `leftDepth=2`, `rightDepth=5` được bao nhiêu?
4. Điều gì sai nếu BFS không chốt `width=len(queue)` ở đầu tầng?
5. Trường hợp nào chiếm nhiều bộ nhớ hơn: cây hình dây với DFS hay cây rất rộng với BFS?
6. Biến thể *đếm lá* khác *đếm tầng* ở phép gộp nào?

**Thử thách nhỏ:** viết lại hai hàm `max_depth_dfs` và `max_depth_bfs` trên trang trắng, rồi tự tạo một cây lệch phải 4 nút để kiểm tra kết quả `4`.

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Có thể tính bằng công thức trên cây con: `depth(node) = 1 + max(depth(left),depth(right))` và `depth(None)=0`. Công thức tự mô tả một bài toán nhỏ hơn.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.